# AutoETF Local Research Agent

这个 Notebook 只面向本地测试。它读取本地 parquet 快照，不依赖 BigQuant 数据环境。

先确保 `data/parquet/local_etf_daily.parquet` 已经存在，再运行下面的聊天单元。

In [1]:
import sys
from dataclasses import replace
from pathlib import Path

for p in [Path.cwd(), Path.cwd().parent]:
    if (p / 'src').exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        project_root = p
        break

from src.config import DEFAULT_CONFIG
from src.data_fetch_akshare import build_local_etf_parquet

local_parquet = project_root / 'data/parquet/local_etf_daily.parquet'
metadata_path = project_root / 'data/parquet/local_etf_metadata.json'

if not local_parquet.exists():
    print('local parquet missing, fetching from AKShare...')
    build_local_etf_parquet(
        symbols=['510300', '159915'],
        start_date='2024-01-01',
        end_date='2024-06-30',
        output_path=str(local_parquet),
        metadata_path=str(metadata_path),
        data_source='auto',
    )
    print('local parquet fetched and saved:', local_parquet)

config = replace(
    DEFAULT_CONFIG,
    data_backend='local',
    local_etf_parquet=str(local_parquet),
    local_benchmark_parquet=None,
)

print('project_root:', project_root)
print('local parquet exists:', local_parquet.exists())


project_root: /Users/yangpeng/Desktop/ai量化
local parquet exists: True


In [2]:
import pandas as pd

path = project_root / "data/parquet/local_etf_daily.parquet"
df = pd.read_parquet(path)

print("shape:", df.shape)
print("columns:")
for i, c in enumerate(df.columns):
    print(i, c)

keywords = [
    "volume", "vol", "amount", "turn", "turnover",
    "ratio", "rate", "money", "close", "open", "high", "low"
]

matched_cols = [
    c for c in df.columns
    if any(k.lower() in c.lower() for k in keywords)
]

print("\nmatched_cols:")
matched_cols

# 换手率字段检查（注意 return_1d 等也会被 turn 命中，需单独筛）
turn_cols = [c for c in df.columns if "turnover" in c.lower() or c.lower() in {"turn", "turnover_rate"}]
print("turnover cols:", turn_cols)

# 拉 159638 最近一天，方便外部核查
code_col = "instrument" if "instrument" in df.columns else "code"
show_cols = [
    "date", code_col, "name", "open", "high", "low", "close",
    "volume", "amount",
    "volume_ratio_5d", "volume_ratio_20d",
    "amount_ratio_5d", "amount_ratio_20d",
    "momentum_20d", "trend_strength", "volatility_20d",
]

sub = df[df[code_col].astype(str) == "159638"].copy()
sub["date"] = pd.to_datetime(sub["date"])
latest = sub["date"].max()
row = sub.loc[sub["date"] == latest, show_cols]

print(f"\n159638 最新交易日: {latest.date()}")
display(row)

print("\n最近 5 个交易日:")
display(sub[show_cols].tail(5))

shape: (95579, 54)
columns:
0 date
1 code
2 instrument
3 name
4 open
5 high
6 low
7 close
8 volume
9 amount
10 source
11 adjust_type
12 prevclose
13 return_1d
14 return_5d
15 momentum_5d
16 momentum_20d
17 momentum_60d
18 future_return_1d
19 future_return_5d
20 future_return_10d
21 future_return_20d
22 ma_5d
23 ma_20d
24 ma_60d
25 ma_gap_20d
26 ma_gap_60d
27 trend_strength
28 trend_persistence_20d
29 volume_ratio_5d
30 volume_ratio_20d
31 amount_ratio_5d
32 amount_ratio_20d
33 crowding_amount_ratio_5d
34 volume_price_divergence_20d
35 volume_breakout_20d
36 volatility_20d
37 volatility_60d
38 risk_adjusted_momentum_20d
39 drawdown_20d
40 drawdown_60d
41 new_high_distance_60d
42 breakout_60d
43 ma_alignment_score
44 reversal_5d
45 ma_deviation_reversal_20d
46 downside_volatility_20d
47 rsi_14d
48 obv_20d
49 obv_trend_20d
50 relative_strength_20d
51 relative_strength_60d
52 benchmark_return_1d
53 beta_60d

matched_cols:
turnover cols: []

159638 最新交易日: 2026-06-25


,date,instrument,name,open,high,low,close,volume,amount,volume_ratio_5d,volume_ratio_20d,amount_ratio_5d,amount_ratio_20d,momentum_20d,trend_strength,volatility_20d
2584,2026-06-25,159638,None,0.92,0.94,0.915,0.926,55809620,51662655,0.910863,0.85592,0.904192,0.845717,-0.058943,-0.001281,0.020041



最近 5 个交易日:


,date,instrument,name,open,high,low,close,volume,amount,volume_ratio_5d,volume_ratio_20d,amount_ratio_5d,amount_ratio_20d,momentum_20d,trend_strength,volatility_20d
2580,2026-06-18,159638,None,0.937,0.961,0.931,0.950,55774704,52987576,1.054966,0.791005,1.068692,0.785961,-0.053785,-0.014274,0.020082
2581,2026-06-22,159638,None,0.949,0.956,0.924,0.956,66135304,62168946,1.275916,0.927584,1.277629,0.914593,-0.060904,-0.005581,0.019822
2582,2026-06-23,159638,None,0.953,0.953,0.915,0.919,58979200,54895697,1.099709,0.840865,1.089372,0.825169,-0.111219,-0.002913,0.020835
2583,2026-06-24,159638,None,0.917,0.933,0.905,0.928,69656900,63969130,1.169234,1.011548,1.148352,0.984943,-0.080278,-0.000532,0.020630
2584,2026-06-25,159638,None,0.920,0.940,0.915,0.926,55809620,51662655,0.910863,0.855920,0.904192,0.845717,-0.058943,-0.001281,0.020041


In [3]:
import getpass
from IPython.display import clear_output

from src.notebook_reload import reload_autoetf_runtime
from src.notebook_chat_ui import launch_notebook_chat
from src.secret_loader import load_deepseek_api_key

reload_autoetf_runtime()

api_key = load_deepseek_api_key(project_root)
if not api_key:
    api_key = getpass.getpass('DeepSeek API Key:')

clear_output(wait=True)

chat_ui = launch_notebook_chat(
    config=config,
    use_llm=True,
    api_key=api_key,
    model='deepseek-v4-flash',
)
print('AutoETF chat UI launched. Re-run this cell to refresh code; duplicate replies should no longer appear.')


AutoETF chat UI launched. Re-run this cell to refresh code; duplicate replies should no longer appear.


In [4]:
from src.pipeline import run_research_pipeline
from IPython.display import Markdown, Image, display

# One-shot research entry: 改下面这一行即可，不要只改 run_research_pipeline 里的字符串。
user_idea = "研究换手率+量比对涨跌概率的关系"

result = run_research_pipeline(
    user_idea=user_idea,
    config=config,
    use_llm=True,
    api_key=api_key,
    outputs_dir=str(project_root / "outputs"),
)

print("任务类型:", result["task_type"])
print("报告已写入:", result["report_path"])
print("图表数量:", len(result.get("plot_paths", [])))
print("耗时: {:.1f}s".format(result["duration_sec"]))

# 在 Notebook 里直接渲染 markdown 报告 + 所有图表
if result["report_markdown"]:
    display(Markdown(result["report_markdown"]))
for p in result.get("plot_paths", []):
    display(Image(filename=p))


RuntimeError: 请先在 cell 4 顶部设置 user_idea，例如: user_idea = '研究 20 日动量'